# Train Word2Vec with a Visible TensorFlow ANN
**One model, one training loop, no Gensim.** Train a real Skip-Gram neural network with negative sampling on WikiText. See the two embedding matrices, forward pass, binary cross-entropy, gradients, and weight updates.

Run cells in order on Google Colab (CPU is fine for the default settings).

In [ ]:
!pip -q install datasets matplotlib scikit-learn

## 1. Download real training text
WikiText-2 is the default. WikiText-103 is much larger and needs substantially more time and RAM.

In [ ]:
from datasets import load_dataset
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.decomposition import PCA

DATASET = 'wikitext-2-raw-v1'  # Change to 'wikitext-103-raw-v1' for a larger corpus.
DIM = 100                      # Numbers in each word vector.
WINDOW = 2                     # Context words on each side.
VOCAB_LIMIT = 12000            # Keep the most frequent words.
MAX_PAIRS = 200000             # Increase for longer training.
EPOCHS = 5

np.random.seed(42)
tf.random.set_seed(42)
raw = load_dataset('Salesforce/wikitext', DATASET, split='train')

# Lowercase and keep alphabetic words only.
sentences = [__import__('re').findall(r"[a-z]+", line.lower()) for line in raw['text']]
sentences = [s for s in sentences if len(s) >= 3]
print('Sentences:', len(sentences))

## 2. Convert words to integer IDs and make training pairs
Each real (center, context) pair gets label 1. A randomly chosen context gets label 0. This is a simplified negative-sampling method; full Word2Vec commonly uses a frequency-adjusted sampling distribution.

In [ ]:
counts = Counter(word for sentence in sentences for word in sentence)
vocab = [word for word, count in counts.most_common(VOCAB_LIMIT) if count >= 3]
word_id = {word: i for i, word in enumerate(vocab)}

# Collect neighboring word pairs from real text.
positive = []
for sentence in sentences:
    ids = [word_id[w] for w in sentence if w in word_id]
    for i, center in enumerate(ids):
        for j in range(max(0, i-WINDOW), min(len(ids), i+WINDOW+1)):
            if i != j:
                positive.append((center, ids[j]))
        if len(positive) >= MAX_PAIRS:
            break
    if len(positive) >= MAX_PAIRS:
        break
positive = np.array(positive[:MAX_PAIRS], dtype='int32')

# Add one random negative for each positive pair.
rng = np.random.default_rng(42)
negative = np.column_stack((positive[:, 0], rng.integers(len(vocab), size=len(positive))))
pairs = np.concatenate([positive, negative])
labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))]).astype('float32')
train_data = tf.data.Dataset.from_tensor_slices((pairs, labels)).shuffle(20000).batch(512)
print('Vocabulary:', len(vocab), '| Positive pairs:', len(positive), '| Negative pairs:', len(negative))

## 3. Build the Word2Vec ANN

`center word ID → input embedding ─┐`

`                                  dot product → sigmoid → loss`

`context word ID → output embedding ┘`

The two `Embedding` layers are **trainable weight matrices**. A dot product scores a word pair. We return a *logit* and let the loss function apply sigmoid numerically safely. There is no additional dense hidden layer.

In [ ]:
# This is the entire neural network.
class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.input_embedding = tf.keras.layers.Embedding(vocab_size, dim)
        self.output_embedding = tf.keras.layers.Embedding(vocab_size, dim)

    def call(self, pairs):
        center = self.input_embedding(pairs[:, 0])
        context = self.output_embedding(pairs[:, 1])
        return tf.reduce_sum(center * context, axis=1)  # Dot-product logits.

model = Word2Vec(len(vocab), DIM)
model(tf.constant([[0, 1]]))  # Create the weights.
model.summary()
print('Input embedding matrix:', model.input_embedding.embeddings.shape)
print('Output embedding matrix:', model.output_embedding.embeddings.shape)

## 4. Train: forward → loss → backpropagation → update
`GradientTape` calculates gradients of the loss with respect to **both** embedding matrices. Adam uses those gradients to update the weights.

In [ ]:
optimizer = tf.keras.optimizers.Adam(0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# Remember one vector so we can see it change.
example_id = word_id.get('king', 0)
before = model.input_embedding.embeddings.numpy()[example_id].copy()
loss_history = []

for epoch in range(EPOCHS):
    losses = []
    for batch_pairs, batch_labels in train_data:
        with tf.GradientTape() as tape:
            logits = model(batch_pairs)                   # 1. Forward pass.
            loss = loss_fn(batch_labels, logits)          # 2. Calculate loss.
        gradients = tape.gradient(loss, model.trainable_variables)  # 3. Backpropagation.
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))  # 4. Update weights.
        losses.append(float(loss))
    loss_history.append(np.mean(losses))
    print(f'Epoch {epoch+1}/{EPOCHS}: loss = {loss_history[-1]:.4f}')

plt.plot(loss_history, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy'); plt.title('ANN training loss'); plt.grid(); plt.show()
after = model.input_embedding.embeddings.numpy()[example_id]
print('Word:', vocab[example_id])
print('First 5 numbers before:', np.round(before[:5], 3))
print('First 5 numbers after: ', np.round(after[:5], 3))

## 5. Test the learned embeddings
Compare nearest words with cosine similarity, then try vector arithmetic. A small corpus and few epochs do **not** guarantee classic analogies.

In [ ]:
vectors = model.input_embedding.embeddings.numpy()
normalized = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-8)

def nearest(vector, exclude=(), k=5):
    scores = normalized @ (vector / max(np.linalg.norm(vector), 1e-8))
    order = np.argsort(-scores)
    return [(vocab[i], round(float(scores[i]), 3)) for i in order if vocab[i] not in exclude][:k]

for word in ['king', 'queen', 'computer', 'science']:
    if word in word_id:
        print(word, nearest(vectors[word_id[word]], exclude={word}))

if all(w in word_id for w in ['king', 'man', 'woman']):
    analogy = vectors[word_id['king']] - vectors[word_id['man']] + vectors[word_id['woman']]
    print('king - man + woman:', nearest(analogy, exclude={'king','man','woman'}))

## 6. Plot 100D vectors in 2D
PCA is only a visualization; distances in this projection may differ from the original 100D space.

In [ ]:
words = [w for w in ['king','queen','man','woman','cat','dog','computer','science','music','city','country','france','paris'] if w in word_id]
if len(words) >= 3:
    points = PCA(n_components=2).fit_transform(np.array([vectors[word_id[w]] for w in words]))
    plt.figure(figsize=(9, 6))
    plt.scatter(points[:, 0], points[:, 1])
    for word, (x, y) in zip(words, points):
        plt.annotate(word, (x, y), xytext=(4, 4), textcoords='offset points')
    plt.title('Learned embeddings (PCA)'); plt.grid(alpha=0.3); plt.show()

## 7. Save your trained ANN and embeddings

In [ ]:
model.save_weights('word2vec_ann.weights.h5')
np.savez('word_vectors.npz', words=np.array(vocab), vectors=vectors)
print('Saved model weights and learned vectors.')

## Student experiments
1. Change `DIM` from 100 to 20 or 200. What happens?
2. Change `WINDOW` from 2 to 5. Do nearest neighbors change?
3. Increase `MAX_PAIRS` and `EPOCHS`. Does the loss decrease?
4. Switch to WikiText-103 if Colab memory permits.
5. Explain exactly which trainable matrix contains the final word vectors.

**Important:** This is a simplified SGNS teaching implementation. Random negative pairs may accidentally include true neighbors; the model learns co-occurrence scores, not guaranteed human-like semantics.